# Module 2 — Traffic Analyzer (Production)

This notebook is the **production pipeline** of VAAET. It combines real-time
perception (YOLO 11 + SORT tracking + speed estimation) with the trained
tabular classifier (TF/Keras MLP) to classify traffic states from video clips.

## Architecture

```mermaid
flowchart LR
    A[Video .mp4] --> B[YOLO 11\nDetection]
    B --> C[SORT\nTracking]
    C --> D[Speed\nEstimation]
    D --> E[Feature\nEngineering\n14 features]
    E --> F[MLP Classifier\ntraffic_classifier.keras]
    F --> G{Traffic State}
    G --> H[(telemetry_raw)]
    G --> I[(traffic_classifications)]
    H & I --> J[Feedback Loop\nRe-training]
    J -.-> F
```

## Traffic States

| State | Code | Criteria |
|---|---|---|
| **Normal** | 0 | Free flow (default) |
| **Reduced** | 1 | Degraded flow: 5–40 km/h, 15–25 veh/min |
| **Congested** | 2 | Congestion: <5 km/h, >25 veh/min, sustained ≥2 min |
| **Accident** | 3 | Disruptive event: ~0 km/h after sudden braking, sustained ≥3 min |

## Prerequisites

- Trained model artifacts in `models/intelligence/` (generated by `notebooks/01_data_prep/data_preparation.ipynb`)
- YOLO 11 weights (downloaded automatically at runtime)

In [ ]:
# Cell 0 — Environment Setup (Colab / Local)

import os
import sys

try:
    import google.colab  # type: ignore[import-untyped]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/titesen/vaaet.git"
    REPO_DIR = "/content/vaaet"
    NB_DIR = os.path.join(REPO_DIR, "notebooks", "02_production")

    if not os.path.isdir(REPO_DIR):
        print("📦 Cloning VAAET repository...")
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    else:
        print("📂 Repository found, pulling latest...")
        os.system(f"git -C {REPO_DIR} pull --ff-only")

    os.chdir(NB_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Colab CWD → {os.getcwd()}")
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✅ Local environment — CWD: {os.getcwd()}")

In [ ]:
# Cell 1 — Dependencies + Load Trained Model

import subprocess

def install_if_missing(package: str, import_name: str | None = None) -> None:
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_if_missing("ultralytics")
install_if_missing("sqlalchemy")
install_if_missing("psycopg2-binary", "psycopg2")

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import joblib
from datetime import datetime

from src.config import (
    FEATURE_COLS, MODEL_DIR, MODEL_PATH, SCALER_PATH, LABEL_MAP_PATH,
    RANDOM_SEED, STATE_LABELS, MODEL_VERSION,
)
from src.features import engineer_features
from src.labeling import assign_traffic_state
from src.db import get_db_config, get_engine
from src.perception.detector import YOLODetector
from src.perception.tracker import SORTTracker
from src.perception.speed import estimate_speed

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Load trained artifacts
_root = os.path.join("..", "..")
model = tf.keras.models.load_model(os.path.join(_root, MODEL_PATH))
scaler = joblib.load(os.path.join(_root, SCALER_PATH))
label_mapping = joblib.load(os.path.join(_root, LABEL_MAP_PATH))

print(f"✅ Dependencies loaded — TF {tf.__version__}")
print(f"✅ Model loaded: {os.path.join(_root, MODEL_PATH)}")
print(f"   Classes: {list(label_mapping.values())}")

## Perception Pipeline

The perception pipeline processes a video clip frame-by-frame:
1. **YOLO 11** detects vehicles (car, truck, bus, motorcycle, bicycle)
2. **SORT tracker** assigns persistent IDs via Euclidean distance matching
3. **Speed estimation** converts pixel displacement to km/h with perspective correction

The output is a per-minute telemetry record with the same schema as `traffic_data` —
this allows the classifier to consume it with identical feature engineering.

In [ ]:
# Cell 2 — Video Input + Perception Pipeline

def process_clip(video_path: str, model_variant: str = "yolo11m") -> pd.DataFrame:
    """Process a video clip and extract per-minute telemetry.

    Args:
        video_path: Path to the .mp4 file.
        model_variant: YOLO model variant to use.

    Returns:
        DataFrame with one row per minute, matching traffic_data schema.
    """
    detector = YOLODetector(model_variant=model_variant)
    detector.load()
    tracker = SORTTracker(max_distance=100, max_lost=30)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_per_minute = int(fps * 60)

    records: list[dict] = []
    frame_idx = 0
    minute_counts = {vtype: 0 for vtype in ("car", "truck", "bus", "motorcycle", "bicycle")}
    minute_speeds: list[float] = []

    print(f"🎬 Processing: {video_path} ({fps:.0f} FPS, {frame_h}p)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Detect
        detections = detector.detect(frame)

        # Track
        det_tuples = [(d.centroid, d.vehicle_type) for d in detections]
        active_tracks = tracker.update(det_tuples)

        # Count vehicles by type
        for det in detections:
            minute_counts[det.vehicle_type] = minute_counts.get(det.vehicle_type, 0) + 1

        # Estimate speed for active tracks
        for track in active_tracks:
            speed = estimate_speed(
                track.history, fps=fps,
                frame_height=frame_h,
            )
            if speed is not None:
                minute_speeds.append(speed)

        frame_idx += 1

        # Aggregate every minute
        if frame_idx % frames_per_minute == 0:
            avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
            total = sum(minute_counts.values())
            records.append({
                "record_time": datetime.now(),
                "avg_speed": round(avg_speed, 2),
                "count_car": minute_counts.get("car", 0),
                "count_truck": minute_counts.get("truck", 0),
                "count_bus": minute_counts.get("bus", 0),
                "count_motorcycle": minute_counts.get("motorcycle", 0),
                "count_bicycle": minute_counts.get("bicycle", 0),
                "total_vehicles": total,
            })
            minute_counts = {vtype: 0 for vtype in minute_counts}
            minute_speeds.clear()
            print(f"   📊 Minute {len(records)}: {avg_speed:.1f} km/h, {total} vehicles")

    cap.release()
    print(f"✅ Clip processed: {len(records)} minute(s) of telemetry")
    return pd.DataFrame(records)


# Execution 
# TODO: Replace with actual video path or upload widget
VIDEO_PATH = "your_video.mp4"  # Set your video path here
# df_telemetry = process_clip(VIDEO_PATH)
print("⚠️ Set VIDEO_PATH and uncomment the line above to process a clip")

## Classification Pipeline

Takes the per-minute telemetry produced by the perception step, applies the
same feature engineering used during training (via `src.features`), and
classifies each record using the pre-trained MLP model.

In [ ]:
# Cell 3 — Feature Engineering + Classification

def classify_telemetry(df_telemetry: pd.DataFrame) -> pd.DataFrame:
    """Apply feature engineering and classify traffic state.

    Args:
        df_telemetry: Raw per-minute telemetry from process_clip().

    Returns:
        DataFrame with 14 features + traffic_state + state_label + confidence.
    """
    # Feature engineering (shared with training — 9 → 14 features)
    df_feat = engineer_features(df_telemetry)

    # Scale features
    X = scaler.transform(df_feat[FEATURE_COLS].values)

    # Predict
    proba = model.predict(X, verbose=0)
    pred_codes = proba.argmax(axis=1)
    confidences = proba.max(axis=1)

    df_feat["traffic_state"] = pred_codes
    df_feat["state_label"] = [label_mapping.get(c, "Unknown") for c in pred_codes]
    df_feat["confidence"] = confidences.round(4)

    print("✅ Classification complete:")
    for code in sorted(df_feat["traffic_state"].unique()):
        count = (df_feat["traffic_state"] == code).sum()
        label = label_mapping.get(code, "Unknown")
        print(f"   {label:>10}: {count} records")

    return df_feat


# Execution 
# TODO: Uncomment after processing a clip in Cell 2
# df_classified = classify_telemetry(df_telemetry)
print("⚠️ Run Cell 2 first, then uncomment the line above")

## Persistence and Feedback

Results are persisted to two PostgreSQL tables:
- **`telemetry_raw`**: The 14 engineered features with FK to original data
- **`traffic_classifications`**: Predictions + confidence + HITL fields

The HITL (Human-in-the-Loop) fields allow operators to validate/override
classifications, creating a feedback loop for model improvement.

In [ ]:
# Cell 4 — Persist Results to Database

from sqlalchemy import text as sa_text

DDL_TELEMETRY_RAW = """
CREATE TABLE IF NOT EXISTS telemetry_raw (
    id SERIAL PRIMARY KEY,
    source_record_id INTEGER REFERENCES traffic_data(id),
    record_time TIMESTAMP NOT NULL,
    avg_speed NUMERIC(5,2),
    total_vehicles INTEGER,
    count_car INTEGER, count_truck INTEGER, count_bus INTEGER,
    count_motorcycle INTEGER, count_bicycle INTEGER,
    heavy_vehicle_ratio NUMERIC(5,4),
    delta_speed NUMERIC(6,2), delta_count INTEGER,
    transition_flag SMALLINT DEFAULT 0,
    speed_variance NUMERIC(6,2),
    hour_of_day SMALLINT, weather_condition SMALLINT DEFAULT 0,
    UNIQUE (source_record_id)
);
"""

DDL_TRAFFIC_CLASSIFICATIONS = """
CREATE TABLE IF NOT EXISTS traffic_classifications (
    id SERIAL PRIMARY KEY,
    telemetry_id INTEGER REFERENCES telemetry_raw(id),
    classified_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    traffic_state SMALLINT NOT NULL,
    state_label TEXT NOT NULL,
    confidence NUMERIC(5,4) NOT NULL,
    model_version TEXT NOT NULL,
    is_human_validated BOOLEAN DEFAULT FALSE,
    human_override_state SMALLINT,
    validated_at TIMESTAMP,
    UNIQUE (telemetry_id, model_version)
);
"""

TELEMETRY_COLS: list[str] = [
    "source_record_id", "record_time", "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle",
    "count_bicycle", "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]

INSERT_TELEMETRY_SQL: str = """
    INSERT INTO telemetry_raw (
        source_record_id, record_time, avg_speed, total_vehicles,
        count_car, count_truck, count_bus, count_motorcycle,
        count_bicycle, heavy_vehicle_ratio, delta_speed, delta_count,
        transition_flag, speed_variance, hour_of_day, weather_condition
    ) VALUES (
        :source_record_id, :record_time, :avg_speed, :total_vehicles,
        :count_car, :count_truck, :count_bus, :count_motorcycle,
        :count_bicycle, :heavy_vehicle_ratio, :delta_speed, :delta_count,
        :transition_flag, :speed_variance, :hour_of_day, :weather_condition
    )
    ON CONFLICT (source_record_id) DO NOTHING
"""

INSERT_CLASSIFICATION_SQL: str = """
    INSERT INTO traffic_classifications (
        telemetry_id, traffic_state, state_label,
        confidence, model_version
    ) VALUES (
        :telemetry_id, :traffic_state, :state_label,
        :confidence, :model_version
    )
    ON CONFLICT (telemetry_id, model_version) DO NOTHING
"""


def persist_classifications(df: pd.DataFrame, config: dict[str, str]) -> None:
    """Persist classified telemetry to PostgreSQL.

    Steps:
      1. Create tables if they don't exist.
      2. Batch INSERT into ``telemetry_raw`` (14 features per record).
      3. Batch INSERT into ``traffic_classifications`` (state + confidence).

    Args:
        df: DataFrame with features + ``traffic_state`` + ``confidence``
            columns.  Must also contain an ``id`` column mapping back to
            ``traffic_data.id`` (used as ``source_record_id``).
        config: Database credentials dict.
    """
    engine = get_engine(config)

    with engine.begin() as conn:
        conn.execute(sa_text(DDL_TELEMETRY_RAW))
        conn.execute(sa_text(DDL_TRAFFIC_CLASSIFICATIONS))
        print("✅ Tables created (or already exist)")

        # ── telemetry_raw batch INSERT ──
        df_telemetry = df.rename(columns={"id": "source_record_id"})[
            [c for c in TELEMETRY_COLS if c in df.columns or c == "source_record_id"]
        ].copy()

        # Ensure correct types for PostgreSQL
        for int_col in ("delta_count", "transition_flag", "hour_of_day", "weather_condition"):
            if int_col in df_telemetry.columns:
                df_telemetry[int_col] = df_telemetry[int_col].astype(int)

        telemetry_records = df_telemetry.to_dict(orient="records")
        conn.execute(sa_text(INSERT_TELEMETRY_SQL), telemetry_records)
        print(f"📊 telemetry_raw: {len(telemetry_records)} records sent")

        # ── Map source_record_id → telemetry_raw.id ──
        telemetry_ids = conn.execute(
            sa_text("SELECT id, source_record_id FROM telemetry_raw ORDER BY id")
        ).fetchall()
        source_to_telemetry = {row[1]: row[0] for row in telemetry_ids}

        # ── traffic_classifications batch INSERT ──
        classification_records: list[dict] = []
        for idx, (_, row) in enumerate(df.iterrows()):
            source_id = row.get("id")
            telemetry_id = source_to_telemetry.get(source_id)
            if telemetry_id is None:
                continue
            classification_records.append({
                "telemetry_id": int(telemetry_id),
                "traffic_state": int(row["traffic_state"]),
                "state_label": STATE_LABELS[int(row["traffic_state"])],
                "confidence": float(round(row.get("confidence", 0.0), 4)),
                "model_version": MODEL_VERSION,
            })

        if classification_records:
            conn.execute(sa_text(INSERT_CLASSIFICATION_SQL), classification_records)

        print(f"📊 traffic_classifications: {len(classification_records)} records sent")

    engine.dispose()

    # Summary
    print(f"\n✅ Persistence completed (model_version={MODEL_VERSION})")
    dist = df["traffic_state"].value_counts().sort_index()
    for code, count in dist.items():
        label = STATE_LABELS.get(code, f"State {code}")
        print(f"   {label:>10}: {count} classifications")


# Execution
try:
    db_config = get_db_config()
    persist_classifications(df_classified, db_config)
except NameError:
    print("⚠️ Run Cells 2-3 first, then re-run this cell")
except Exception as e:
    print(f"🔴 Persistence error: {e}")
    print("   Classified data is available in-memory (df_classified)")

## Feedback Loop — Re-training

This cell implements the self-improvement cycle:
1. Load human-validated records from `traffic_classifications` (where `is_human_validated = TRUE`)
2. Merge with original training data
3. Re-train the MLP with the expanded dataset
4. Export updated `.keras` artifact

This closes the feedback loop: **production → HITL validation → re-training → better production**.

In [ ]:
# Cell 5 — Feedback Loop: Re-train with HITL Data (Optional)

from sklearn.preprocessing import StandardScaler as _StandardScaler
from sklearn.model_selection import train_test_split as _train_test_split
from sklearn.metrics import f1_score as _f1_score
from imblearn.over_sampling import SMOTE as _SMOTE


def retrain_with_feedback(config: dict[str, str]) -> None:
    """Re-train the classifier using human-validated data.

    Loads validated classifications from the database, merges them with
    the original training data, and re-trains the MLP model. The updated
    model is exported only if F1-macro improves over the current one.

    Args:
        config: Database credentials.
    """
    engine = get_engine(config)

    # ── Load human-validated records ──
    query = """
        SELECT tr.*, tc.traffic_state AS validated_state
        FROM telemetry_raw tr
        JOIN traffic_classifications tc ON tc.telemetry_id = tr.id
        WHERE tc.is_human_validated = TRUE
        ORDER BY tr.record_time
    """
    df_validated = pd.read_sql(sa_text(query), engine)
    engine.dispose()

    if df_validated.empty:
        print("⚠️ No human-validated records found. Skipping re-training.")
        return

    print(f"📊 Loaded {len(df_validated)} validated records")

    # ── Merge with original training data ──
    _root = os.path.join("..", "..")
    csv_path = os.path.join(_root, "data", "processed", "traffic_telemetry.csv")
    if not os.path.exists(csv_path):
        print("🔴 Original training CSV not found. Run Module 1 first.")
        return

    df_original = pd.read_csv(csv_path)
    if "traffic_state" not in df_original.columns:
        df_original["traffic_state"] = assign_traffic_state(df_original)

    # Use validated_state as ground truth for HITL records
    df_validated_features = df_validated[FEATURE_COLS].copy()
    df_validated_features["traffic_state"] = df_validated["validated_state"].astype(int)

    df_combined = pd.concat(
        [df_original[FEATURE_COLS + ["traffic_state"]], df_validated_features],
        ignore_index=True,
    )
    print(f"📊 Combined dataset: {len(df_combined)} records "
          f"({len(df_original)} original + {len(df_validated)} validated)")

    # ── Split (raw, unscaled) ──
    X_raw = df_combined[FEATURE_COLS].values
    y = df_combined["traffic_state"].values

    X_train_raw, X_test_raw, y_train, y_test = _train_test_split(
        X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED,
    )

    # ── Evaluate old model on test set (using its own scaler) ──
    X_test_old = scaler.transform(X_test_raw)
    y_pred_old = model.predict(X_test_old, verbose=0).argmax(axis=1)
    f1_old = _f1_score(y_test, y_pred_old, average="macro", zero_division=0)

    # ── Fit new scaler + SMOTE on training data ──
    new_scaler = _StandardScaler()
    X_train_new = new_scaler.fit_transform(X_train_raw)
    X_test_new = new_scaler.transform(X_test_raw)

    train_counts = np.bincount(y_train)
    min_class = train_counts[train_counts > 0].min()
    k_neighbors = min(5, min_class - 1) if min_class > 1 else 1

    if min_class >= 2:
        sm = _SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
        X_train_res, y_train_res = sm.fit_resample(X_train_new, y_train)
        print(f"✅ SMOTE applied (k_neighbors={k_neighbors})")
    else:
        X_train_res, y_train_res = X_train_new, y_train
        print("⚠️ SMOTE skipped — class with <2 samples")

    # ── Re-train MLP ──
    from tensorflow.keras.models import Sequential as _Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
    from tensorflow.keras.callbacks import EarlyStopping

    n_classes = len(np.unique(y))
    new_model = _Sequential([
        Input(shape=(X_train_res.shape[1],)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(n_classes, activation="softmax"),
    ], name="traffic_state_classifier_retrained")

    new_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    new_model.fit(
        X_train_res, y_train_res,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
        verbose=1,
    )

    # ── Evaluate new model on test set ──
    y_pred_new = new_model.predict(X_test_new, verbose=0).argmax(axis=1)
    f1_new = _f1_score(y_test, y_pred_new, average="macro", zero_division=0)

    print(f"\n📊 F1-macro comparison:")
    print(f"   Current model: {f1_old:.4f}")
    print(f"   Retrained:     {f1_new:.4f}")

    # ── Export if improved ──
    if f1_new > f1_old:
        model_path = os.path.join(_root, "models", "intelligence", "traffic_classifier.keras")
        scaler_path = os.path.join(_root, "models", "intelligence", "feature_scaler.joblib")
        new_model.save(model_path)
        joblib.dump(new_scaler, scaler_path)
        print(f"✅ Improved model exported → {model_path}")
        print(f"   New scaler exported → {scaler_path}")
    else:
        print("⚠️ Retrained model did not improve. Keeping current model.")


# Execution
# Uncomment when HITL data is available in traffic_classifications:
# retrain_config = get_db_config()
# retrain_with_feedback(retrain_config)
print("⚠️ Re-training requires human-validated data in traffic_classifications")

## Visualization

Summary dashboard showing traffic state distribution, speed timeline, and
classification confidence. Only runs after Cells 2-3 have been executed.

In [ ]:
# Cell 6 — Visualization Dashboard

import matplotlib.pyplot as plt

def show_dashboard(df: pd.DataFrame) -> None:
    """Display a summary dashboard for the classified telemetry.

    Args:
        df: Classified DataFrame with traffic_state, avg_speed, confidence.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]

    # State distribution
    dist = df["traffic_state"].value_counts().sort_index()
    state_names = [label_mapping.get(c, f"State {c}") for c in sorted(dist.index)]
    state_colors = [colors[c] for c in sorted(dist.index)]
    axes[0].bar(state_names, dist.values, color=state_colors)
    axes[0].set_title("Traffic State Distribution")
    axes[0].set_ylabel("Records")

    # Speed timeline
    axes[1].plot(range(len(df)), df["avg_speed"], color="#3498db", linewidth=1)
    axes[1].set_title("Average Speed Over Time")
    axes[1].set_xlabel("Minute")
    axes[1].set_ylabel("Speed (km/h)")
    axes[1].grid(True, alpha=0.3)

    # Confidence distribution
    axes[2].hist(df["confidence"], bins=20, color="#9b59b6", edgecolor="white")
    axes[2].set_title("Classification Confidence")
    axes[2].set_xlabel("Confidence")
    axes[2].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()


# Execution
# TODO: Uncomment after classification
# show_dashboard(df_classified)
print("⚠️ Run Cells 2-3 first, then uncomment the line above")